#  Feature Extraction

It applies the combined HOG and RGB colour feature extractor to the full dataset.

Each image is converted into one fixed-length feature vector with shape `(6180,)`. The extracted features and labels are then saved for later classifier training.

## 1. Import libraries

Import the libraries needed for loading images, processing dataset files, extracting features, and saving the final arrays.

In [16]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog
from tqdm import tqdm

## 2. Load the shared project configuration

Find the project root and import the shared paths and image settings from `src/config.py`.

In [17]:
current_path = Path.cwd().resolve()

for parent in [current_path, *current_path.parents]:
    if (parent / "src" / "config.py").exists():
        project_root = parent
        break
else:
    raise FileNotFoundError(
        "Could not find the project root containing src/config.py"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config

print("Project root:", project_root)
print("Image size:", config.IMG_SIZE)
print("Raw data root:", config.DATA_RAW_ROOT)
print("Training CSV:", config.TRAIN_CSV)

Project root: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group
Image size: (224, 224)
Raw data root: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\data\raw
Training CSV: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\data\processed\train.csv


## 3. Set the output folder

Create a folder for the extracted traditional features.

The output files will be saved separately from the raw images.

In [18]:
FEATURE_OUTPUT_DIR = (
    project_root
    / "outputs"
    / "traditional_features"
)

FEATURE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Feature output folder:", FEATURE_OUTPUT_DIR)

Feature output folder: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\outputs\traditional_features


## 4. Define the HOG feature function

It resizes the image, converts it to grayscale, and extracts the HOG feature.

The output is a `float32` feature vector with shape `(6084,)`.

In [19]:
def extract_hog_feature(image):
    if not isinstance(image, Image.Image):
        raise TypeError(
            "image must be a PIL.Image.Image object."
        )

    resized_image = image.convert("RGB").resize(
        config.IMG_SIZE
    )

    image_array = np.asarray(
        resized_image
    )

    gray_image = np.dot(
        image_array[..., :3],
        [0.299, 0.587, 0.114]
    )

    hog_feature = hog(
        gray_image,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )

    return hog_feature.astype(np.float32)

## 5. Define the RGB colour histogram function

It calculates a normalised 32-bin histogram for each RGB channel.

The three histograms are combined into a `float32` feature vector with shape `(96,)`.

In [20]:
def extract_colour_feature(image, bins=32):
    if not isinstance(image, Image.Image):
        raise TypeError(
            "image must be a PIL.Image.Image object."
        )

    if not isinstance(bins, int) or bins <= 0:
        raise ValueError(
            "bins must be a positive integer."
        )

    resized_image = image.convert("RGB").resize(
        config.IMG_SIZE
    )

    image_array = np.asarray(
        resized_image
    )

    channel_histograms = []

    for channel_index in range(3):
        histogram, _ = np.histogram(
            image_array[:, :, channel_index],
            bins=bins,
            range=(0, 256)
        )

        histogram = histogram.astype(
            np.float32
        )

        histogram_sum = histogram.sum()

        if histogram_sum > 0:
            histogram = histogram / histogram_sum

        channel_histograms.append(
            histogram
        )

    colour_feature = np.concatenate(
        channel_histograms
    )

    return colour_feature.astype(np.float32)

## 6. Define the combined feature function

It combines the HOG feature and RGB colour histogram.

Each image produces one `float32` feature vector with shape `(6180,)`.

In [21]:
def extract_combined_feature(image):
    hog_feature = extract_hog_feature(
        image
    )

    colour_feature = extract_colour_feature(
        image,
        bins=32
    )

    combined_feature = np.concatenate(
        [
            hog_feature,
            colour_feature
        ]
    )

    return combined_feature.astype(
        np.float32
    )

## 7. Load the dataset CSV files

Load the training, validation, and test CSV files and check their sizes and columns.

In [22]:
csv_files = {
    "train": Path(config.TRAIN_CSV),
    "validation": Path(config.VAL_CSV),
    "test": Path(config.TEST_CSV)
}

dataset_tables = {}

for split_name, csv_path in csv_files.items():
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{split_name} CSV was not found: {csv_path}"
        )

    dataset_df = pd.read_csv(
        csv_path
    )

    dataset_tables[split_name] = dataset_df

    print(f"{split_name}:")
    print("CSV path:", csv_path)
    print("Number of rows:", len(dataset_df))
    print("Columns:", dataset_df.columns.tolist())
    print()

train:
CSV path: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\data\processed\train.csv
Number of rows: 20000
Columns: ['file_path', 'label', 'category_id', 'category_name']

validation:
CSV path: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\data\processed\val.csv
Number of rows: 5000
Columns: ['file_path', 'label', 'category_id', 'category_name']

test:
CSV path: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\data\processed\test.csv
Number of rows: 5000
Columns: ['file_path', 'label', 'category_id', 'category_name']



## 8. Check the required columns

Each dataset CSV must contain an image path column and a label column.

It expects these columns to be named `file_path` and `label`.

In [23]:
required_columns = {
    "file_path",
    "label"
}

for split_name, dataset_df in dataset_tables.items():
    missing_columns = (
        required_columns
        - set(dataset_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{split_name} CSV is missing columns: "
            f"{sorted(missing_columns)}"
        )

    if dataset_df.empty:
        raise ValueError(
            f"{split_name} CSV is empty."
        )

    print(
        f"{split_name}: required columns found"
    )

train: required columns found
validation: required columns found
test: required columns found


## 9. Test one image

Load the first training image and check that the combined feature extractor produces the expected output.

In [24]:
sample_row = dataset_tables["train"].iloc[0]

sample_image_path = (
    Path(config.DATA_RAW_ROOT)
    / str(sample_row["file_path"])
)

if not sample_image_path.exists():
    raise FileNotFoundError(
        f"Sample image was not found: "
        f"{sample_image_path}"
    )

with Image.open(sample_image_path) as image:
    sample_image = image.convert("RGB")

sample_feature = extract_combined_feature(
    sample_image
)

print("Sample image:", sample_image_path)
print("Label:", sample_row["label"])
print("Feature shape:", sample_feature.shape)
print("Feature dtype:", sample_feature.dtype)
print(
    "Contains NaN:",
    np.isnan(sample_feature).any()
)
print(
    "Contains infinity:",
    np.isinf(sample_feature).any()
)

Sample image: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\data\raw\train_mini\07078_Plantae_Tracheophyta_Magnoliopsida_Asterales_Asteraceae_Xylorhiza_orcuttii\f670b8a9-7f74-4424-9b2c-5ed900868df4.jpg
Label: 364
Feature shape: (6180,)
Feature dtype: float32
Contains NaN: False
Contains infinity: False


## 10. Validate the sample feature

Check the feature length, data type, and values before processing the full dataset.

In [25]:
sample_checks = {
    "Feature is one-dimensional":
        sample_feature.ndim == 1,

    "Feature length is 6180":
        sample_feature.shape == (6180,),

    "Feature dtype is float32":
        sample_feature.dtype == np.float32,

    "Feature contains no NaN":
        not np.isnan(sample_feature).any(),

    "Feature contains no infinity":
        not np.isinf(sample_feature).any()
}

for check_name, passed in sample_checks.items():
    print(f"{check_name}: {passed}")

print(
    "\nAll sample checks passed:",
    all(sample_checks.values())
)

Feature is one-dimensional: True
Feature length is 6180: True
Feature dtype is float32: True
Feature contains no NaN: True
Feature contains no infinity: True

All sample checks passed: True


## 11. Define the dataset feature extraction function

It loads each image in a dataset split and extracts one combined feature vector.

It returns:

- a feature matrix
- an array of labels
- an array of image paths

In [26]:
def extract_dataset_features(
    dataset_df,
    split_name
):
    if not isinstance(dataset_df, pd.DataFrame):
        raise TypeError(
            "dataset_df must be a pandas DataFrame."
        )

    if dataset_df.empty:
        raise ValueError(
            f"{split_name} dataset is empty."
        )

    features = []
    labels = []
    file_paths = []

    for row_index, row in tqdm(
        dataset_df.iterrows(),
        total=len(dataset_df),
        desc=f"Extracting {split_name} features"
    ):
        relative_path = str(
            row["file_path"]
        )

        image_path = (
            Path(config.DATA_RAW_ROOT)
            / relative_path
        )

        if not image_path.exists():
            raise FileNotFoundError(
                f"Image was not found at row "
                f"{row_index}: {image_path}"
            )

        try:
            with Image.open(image_path) as image:
                rgb_image = image.convert("RGB")

                feature = extract_combined_feature(
                    rgb_image
                )

        except Exception as error:
            raise RuntimeError(
                f"Failed to process image at row "
                f"{row_index}: {image_path}"
            ) from error

        if feature.shape != (6180,):
            raise ValueError(
                f"Unexpected feature shape at row "
                f"{row_index}: {feature.shape}"
            )

        if not np.isfinite(feature).all():
            raise ValueError(
                f"Invalid feature values at row "
                f"{row_index}: {image_path}"
            )

        features.append(
            feature
        )

        labels.append(
            row["label"]
        )

        file_paths.append(
            relative_path
        )

    features = np.stack(
        features
    ).astype(np.float32)

    labels = np.asarray(
        labels
    )

    file_paths = np.asarray(
        file_paths
    )

    return features, labels, file_paths

## 12. Run a small test

Process 10 training images before extracting features from the complete dataset.

In [27]:
small_test_df = (
    dataset_tables["train"]
    .head(10)
    .copy()
)

X_small, y_small, paths_small = (
    extract_dataset_features(
        small_test_df,
        split_name="small test"
    )
)

print("Small feature shape:", X_small.shape)
print("Small label shape:", y_small.shape)
print("Small path shape:", paths_small.shape)
print("Feature dtype:", X_small.dtype)

print(
    "All values are finite:",
    np.isfinite(X_small).all()
)

Extracting small test features: 100%|██████████| 10/10 [00:00<00:00, 84.67it/s]

Small feature shape: (10, 6180)
Small label shape: (10,)
Small path shape: (10,)
Feature dtype: float32
All values are finite: True


## 13. Extract training features

Apply the combined feature extractor to the complete training dataset.

In [28]:
X_train, y_train, train_paths = (
    extract_dataset_features(
        dataset_tables["train"],
        split_name="training"
    )
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("train_paths shape:", train_paths.shape)
print("X_train dtype:", X_train.dtype)

Extracting training features: 100%|██████████| 20000/20000 [04:11<00:00, 79.51it/s]


X_train shape: (20000, 6180)
y_train shape: (20000,)
train_paths shape: (20000,)
X_train dtype: float32


## 14. Extract validation features

Apply the combined feature extractor to the complete validation dataset.

In [29]:
X_validation, y_validation, validation_paths = (
    extract_dataset_features(
        dataset_tables["validation"],
        split_name="validation"
    )
)

print(
    "X_validation shape:",
    X_validation.shape
)

print(
    "y_validation shape:",
    y_validation.shape
)

print(
    "validation_paths shape:",
    validation_paths.shape
)

print(
    "X_validation dtype:",
    X_validation.dtype
)

Extracting validation features: 100%|██████████| 5000/5000 [00:59<00:00, 84.34it/s]


X_validation shape: (5000, 6180)
y_validation shape: (5000,)
validation_paths shape: (5000,)
X_validation dtype: float32


## 15. Extract test features

Apply the combined feature extractor to the complete test dataset.

In [30]:
X_test, y_test, test_paths = (
    extract_dataset_features(
        dataset_tables["test"],
        split_name="test"
    )
)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("test_paths shape:", test_paths.shape)
print("X_test dtype:", X_test.dtype)

Extracting test features: 100%|██████████| 5000/5000 [01:34<00:00, 52.91it/s]


X_test shape: (5000, 6180)
y_test shape: (5000,)
test_paths shape: (5000,)
X_test dtype: float32


## 16. Check the extracted datasets

Confirm that all dataset splits have the correct feature length, data type, and number of labels.

In [31]:
dataset_results = {
    "train": (
        X_train,
        y_train,
        train_paths
    ),

    "validation": (
        X_validation,
        y_validation,
        validation_paths
    ),

    "test": (
        X_test,
        y_test,
        test_paths
    )
}

for split_name, result in dataset_results.items():
    features, labels, paths = result

    print(split_name)
    print("Feature shape:", features.shape)
    print("Label shape:", labels.shape)
    print("Path shape:", paths.shape)
    print("Feature dtype:", features.dtype)

    print(
        "Feature length is 6180:",
        features.shape[1] == 6180
    )

    print(
        "Counts match:",
        len(features)
        == len(labels)
        == len(paths)
    )

    print(
        "All values are finite:",
        np.isfinite(features).all()
    )

    print()

train
Feature shape: (20000, 6180)
Label shape: (20000,)
Path shape: (20000,)
Feature dtype: float32
Feature length is 6180: True
Counts match: True
All values are finite: True

validation
Feature shape: (5000, 6180)
Label shape: (5000,)
Path shape: (5000,)
Feature dtype: float32
Feature length is 6180: True
Counts match: True
All values are finite: True

test
Feature shape: (5000, 6180)
Label shape: (5000,)
Path shape: (5000,)
Feature dtype: float32
Feature length is 6180: True
Counts match: True
All values are finite: True



## 17. Check the class distribution

Display the number of images for each label in the training, validation, and test datasets.

In [32]:
for split_name, dataset_df in dataset_tables.items():
    print(f"{split_name} label distribution:")

    label_counts = (
        dataset_df["label"]
        .value_counts()
        .sort_index()
        .rename("image_count")
        .to_frame()
    )

    display(label_counts)

train label distribution:


,image_count
label,
0,40
1,40
2,40
3,40
4,40
...,...
495,40
496,40
497,40


validation label distribution:


,image_count
label,
0,10
1,10
2,10
3,10
4,10
...,...
495,10
496,10
497,10


test label distribution:


,image_count
label,
0,10
1,10
2,10
3,10
4,10
...,...
495,10
496,10
497,10


## 18. Save the extracted features

Save the features, labels, and image paths for each dataset split.

The output files are saved in `outputs/traditional_features`.

In [33]:
train_output_path = (
    FEATURE_OUTPUT_DIR
    / "train_combined_features.npz"
)

validation_output_path = (
    FEATURE_OUTPUT_DIR
    / "validation_combined_features.npz"
)

test_output_path = (
    FEATURE_OUTPUT_DIR
    / "test_combined_features.npz"
)

np.savez_compressed(
    train_output_path,
    features=X_train,
    labels=y_train,
    file_paths=train_paths
)

np.savez_compressed(
    validation_output_path,
    features=X_validation,
    labels=y_validation,
    file_paths=validation_paths
)

np.savez_compressed(
    test_output_path,
    features=X_test,
    labels=y_test,
    file_paths=test_paths
)

print("Saved:", train_output_path)
print("Saved:", validation_output_path)
print("Saved:", test_output_path)

Saved: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\outputs\traditional_features\train_combined_features.npz
Saved: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\outputs\traditional_features\validation_combined_features.npz
Saved: C:\Users\1\Desktop\COMP9517\Project\9517_assignment_MVP_Group\outputs\traditional_features\test_combined_features.npz


## 19. Check the saved files

Reload the saved files and check their shapes and file sizes.

In [34]:
saved_files = {
    "train": train_output_path,
    "validation": validation_output_path,
    "test": test_output_path
}

for split_name, saved_path in saved_files.items():
    if not saved_path.exists():
        raise FileNotFoundError(
            f"Saved file was not found: {saved_path}"
        )

    with np.load(
        saved_path,
        allow_pickle=True
    ) as loaded_data:
        loaded_features = loaded_data["features"]
        loaded_labels = loaded_data["labels"]
        loaded_paths = loaded_data["file_paths"]

        print(split_name)
        print(
            "Loaded feature shape:",
            loaded_features.shape
        )

        print(
            "Loaded label shape:",
            loaded_labels.shape
        )

        print(
            "Loaded path shape:",
            loaded_paths.shape
        )

        print(
            "File size:",
            f"{saved_path.stat().st_size / (1024 ** 2):.2f} MB"
        )

        print()

train
Loaded feature shape: (20000, 6180)
Loaded label shape: (20000,)
Loaded path shape: (20000,)
File size: 370.90 MB

validation
Loaded feature shape: (5000, 6180)
Loaded label shape: (5000,)
Loaded path shape: (5000,)
File size: 92.73 MB

test
Loaded feature shape: (5000, 6180)
Loaded label shape: (5000,)
Loaded path shape: (5000,)
File size: 92.70 MB



## 20. Compare the saved and original data

Confirm that the saved files contain the same feature values, labels, and image paths as the original arrays.

In [35]:
save_checks = {}

for split_name, saved_path in saved_files.items():
    original_features, original_labels, original_paths = (
        dataset_results[split_name]
    )

    with np.load(
        saved_path,
        allow_pickle=True
    ) as loaded_data:
        save_checks[
            f"{split_name} features match"
        ] = np.array_equal(
            loaded_data["features"],
            original_features
        )

        save_checks[
            f"{split_name} labels match"
        ] = np.array_equal(
            loaded_data["labels"],
            original_labels
        )

        save_checks[
            f"{split_name} paths match"
        ] = np.array_equal(
            loaded_data["file_paths"],
            original_paths
        )

for check_name, passed in save_checks.items():
    print(f"{check_name}: {passed}")

print(
    "\nAll saved data checks passed:",
    all(save_checks.values())
)

train features match: True
train labels match: True
train paths match: True
validation features match: True
validation labels match: True
validation paths match: True
test features match: True
test labels match: True
test paths match: True

All saved data checks passed: True


## 21. Final validation

Perform the final checks on all extracted feature matrices and saved output files.

In [36]:
final_checks = {
    "Training feature length is 6180":
        X_train.shape[1] == 6180,

    "Validation feature length is 6180":
        X_validation.shape[1] == 6180,

    "Test feature length is 6180":
        X_test.shape[1] == 6180,

    "Training dtype is float32":
        X_train.dtype == np.float32,

    "Validation dtype is float32":
        X_validation.dtype == np.float32,

    "Test dtype is float32":
        X_test.dtype == np.float32,

    "Training values are finite":
        np.isfinite(X_train).all(),

    "Validation values are finite":
        np.isfinite(X_validation).all(),

    "Test values are finite":
        np.isfinite(X_test).all(),

    "Training counts match":
        len(X_train)
        == len(y_train)
        == len(train_paths),

    "Validation counts match":
        len(X_validation)
        == len(y_validation)
        == len(validation_paths),

    "Test counts match":
        len(X_test)
        == len(y_test)
        == len(test_paths),

    "All output files exist":
        all(
            output_path.exists()
            for output_path in saved_files.values()
        )
}

for check_name, passed in final_checks.items():
    print(f"{check_name}: {passed}")

print(
    "\nAll final checks passed:",
    all(final_checks.values())
)

Training feature length is 6180: True
Validation feature length is 6180: True
Test feature length is 6180: True
Training dtype is float32: True
Validation dtype is float32: True
Test dtype is float32: True
Training values are finite: True
Validation values are finite: True
Test values are finite: True
Training counts match: True
Validation counts match: True
Test counts match: True
All output files exist: True

All final checks passed: True


## Summary

The combined HOG and RGB colour features were extracted from the complete dataset.

Each image was converted into a fixed-length `float32` feature vector with shape `(6180,)`.

The extracted data was saved in:

```text
outputs/traditional_features/